esses são os extratos da conta PJ

In [63]:
import pandas as pd
path = r'C:\Users\nadia\OneDrive\Desktop\Leticia\Documentos\Projetos\cliente_confeitaria\Analise_financeira_confeitaria\Dados\originais\extratos_pagbank'


In [64]:
fev25= pd.read_csv(path+r'\Extrato PagBank fevereiro.csv', skiprows=4,sep=';', encoding='latin-1') # desconsiderando as primerias linhas para trazer apenaas aa partir dos cabeçalho 
fev25.dtypes


Data         str
Descrição    str
Valor        str
Saldo        str
dtype: object

In [65]:
fev25[fev25.isnull().any(axis=1)] # verificando se existem linhas com valores nulos
fev25[fev25.isnull().all(axis=1)]

fev25.duplicated().sum()#linhas duplicadas

np.int64(0)

In [66]:
#verificação de formato de datas e valores
print(fev25["Data"].str.replace(r"\d","9",regex = True).value_counts(dropna = False)) #verificando se todas as datas estão no mesmo formato
print(fev25["Valor"].str.replace(r"\d","9",regex = True).value_counts(dropna = False)) #verificando se todos os valores estão no mesmo formato
print(fev25["Saldo"].str.replace(r"\d","9",regex = True).value_counts(dropna = False)) #verificando se todos os valores estão no mesmo formato
#todos com virgula e datas com /; nao precisará de padronização nesse arquivo

Data
99/99/9999    114
Name: count, dtype: int64
Valor
999,99      80
-999,99     14
-9999,99    12
9999,99      3
-99,99       2
99,99        2
99999,99     1
Name: count, dtype: int64
Saldo
99999,99    71
9999,99     43
Name: count, dtype: int64


In [67]:
extrato= fev25 #adicionando ao extrato completo
extrato 

,Data,Descrição,Valor,Saldo
0,01/02/2025,VENDAS CARTAO CREDITO - REPASSE,"260,63","11401,26"
1,01/02/2025,VENDAS CARTAO DEBITO - REPASSE,"159,74","11561,00"
2,01/02/2025,VENDAS PIX MAQUININHA - REPASSE,"436,33","11997,33"
3,01/02/2025,PIX RECEBIDO - ALINE VIEIRA,"265,00","12262,33"
4,02/02/2025,VENDAS CARTAO CREDITO - REPASSE,"245,60","12507,93"
...,...,...,...,...
109,27/02/2025,VENDAS PIX MAQUININHA - REPASSE,"404,38","50430,11"
110,27/02/2025,PAGAMENTO BOLETO - LATICINIOS VALE VERDE LTDA,"-791,03","49639,08"
111,28/02/2025,VENDAS CARTAO CREDITO - REPASSE,"231,42","49870,50"
112,28/02/2025,VENDAS CARTAO DEBITO - REPASSE,"148,36","50018,86"


In [68]:
#tratando separadamente o extrato de março de 26 pois ele é o unico fora do padrão dos outros

mar26= pd.read_csv(path+r'\extrato_mar26_site.csv',sep=',', encoding='latin-1',dtype=str) # desconsiderando as primerias linhas para trazer apenaas aa partir dos cabeçalho 
mar26 

,DATA_LANCAMENTO,HISTORICO,VALOR,TIPO
0,2026-03-01,VENDAS CARTAO CREDITO - REPASSE,348.58,C
1,2026-03-01,VENDAS CARTAO DEBITO - REPASSE,276.22,C
2,2026-03-01,VENDAS PIX MAQUININHA - REPASSE,709.54,C
3,2026-03-01,PIX RECEBIDO - LUANA COSTA,640.00,C
4,2026-03-02,VENDAS CARTAO CREDITO - REPASSE,315.60,C
...,...,...,...,...
147,2026-03-31,VENDAS PIX MAQUININHA - REPASSE,423.30,C
148,2026-03-31,PIX RECEBIDO - DIEGO MACHADO,455.00,C
149,2026-03-31,PIX RECEBIDO - GABRIELA NUNES,235.00,C
150,2026-03-31,PIX RECEBIDO - BRUNO TAVARES,125.00,C


In [69]:
mar26[mar26.isnull().any(axis=1)] # verificando se existem linhas com valores nulos
mar26[mar26.isnull().all(axis=1)]

mar26.duplicated().sum()#linhas duplicadas

np.int64(0)

In [70]:
#verificação de formato de datas e valores
print(mar26["DATA_LANCAMENTO"].str.replace(r"\d","9",regex = True).value_counts(dropna = False)) #verificando se todas as datas estão no mesmo formato
print(mar26["VALOR"].str.replace(r"\d","9",regex = True).value_counts(dropna = False)) #verificando se todos os valores estão no mesmo formato

# aqui não tem valores positivos ou negativos ou negativos. Necessário padronizar de acordo com o tipo "C" ou "D"

DATA_LANCAMENTO
9999-99-99    152
Name: count, dtype: int64
VALOR
999.99     130
9999.99     18
99.99        4
Name: count, dtype: int64


In [71]:
#renomeando as colunas paraa padronização
mar26 = mar26.rename(columns={
    "DATA_LANCAMENTO": "Data",
    "HISTORICO": "Descrição",
    "VALOR": "Valor",
})
mar26

,Data,Descrição,Valor,TIPO
0,2026-03-01,VENDAS CARTAO CREDITO - REPASSE,348.58,C
1,2026-03-01,VENDAS CARTAO DEBITO - REPASSE,276.22,C
2,2026-03-01,VENDAS PIX MAQUININHA - REPASSE,709.54,C
3,2026-03-01,PIX RECEBIDO - LUANA COSTA,640.00,C
4,2026-03-02,VENDAS CARTAO CREDITO - REPASSE,315.60,C
...,...,...,...,...
147,2026-03-31,VENDAS PIX MAQUININHA - REPASSE,423.30,C
148,2026-03-31,PIX RECEBIDO - DIEGO MACHADO,455.00,C
149,2026-03-31,PIX RECEBIDO - GABRIELA NUNES,235.00,C
150,2026-03-31,PIX RECEBIDO - BRUNO TAVARES,125.00,C


In [72]:
#padronização de valor negativo paraa débito. Assim mantém o padrão das outras bases de extrato.


debito = mar26["TIPO"].str.strip().str.upper().str.startswith("D") #responde se o tecto começa com "D"
mar26.loc[debito, "Valor"] = "-" + mar26.loc[debito, "Valor"].str.strip()# concatena o - caso atenda o D
mar26 = mar26.drop(columns=["TIPO"])# remove a coluna tipo

mar26

,Data,Descrição,Valor
0,2026-03-01,VENDAS CARTAO CREDITO - REPASSE,348.58
1,2026-03-01,VENDAS CARTAO DEBITO - REPASSE,276.22
2,2026-03-01,VENDAS PIX MAQUININHA - REPASSE,709.54
3,2026-03-01,PIX RECEBIDO - LUANA COSTA,640.00
4,2026-03-02,VENDAS CARTAO CREDITO - REPASSE,315.60
...,...,...,...
147,2026-03-31,VENDAS PIX MAQUININHA - REPASSE,423.30
148,2026-03-31,PIX RECEBIDO - DIEGO MACHADO,455.00
149,2026-03-31,PIX RECEBIDO - GABRIELA NUNES,235.00
150,2026-03-31,PIX RECEBIDO - BRUNO TAVARES,125.00


In [73]:
print(mar26["Valor"].str.replace(r"\d","9",regex = True).value_counts(dropna = False)) #verificando se todos os valores estão no mesmo formato

Valor
999.99      118
-9999.99     17
-999.99      12
-99.99        2
99.99         2
9999.99       1
Name: count, dtype: int64


In [74]:
mar26["Saldo"]=0
mar26

,Data,Descrição,Valor,Saldo
0,2026-03-01,VENDAS CARTAO CREDITO - REPASSE,348.58,0
1,2026-03-01,VENDAS CARTAO DEBITO - REPASSE,276.22,0
2,2026-03-01,VENDAS PIX MAQUININHA - REPASSE,709.54,0
3,2026-03-01,PIX RECEBIDO - LUANA COSTA,640.00,0
4,2026-03-02,VENDAS CARTAO CREDITO - REPASSE,315.60,0
...,...,...,...,...
147,2026-03-31,VENDAS PIX MAQUININHA - REPASSE,423.30,0
148,2026-03-31,PIX RECEBIDO - DIEGO MACHADO,455.00,0
149,2026-03-31,PIX RECEBIDO - GABRIELA NUNES,235.00,0
150,2026-03-31,PIX RECEBIDO - BRUNO TAVARES,125.00,0


In [75]:
mar26["Valor"] = mar26["Valor"].str.replace(".", ",", regex=False) #conversão de ponto para vírgula para manter o padrão dos extratos

Tratamento e junção de todos os arquivos

In [76]:
#verificação se todos os arquivos estão com valores no padrão de virgula, para o padrão de brasileiro
#necessária a verificação antes de unir os arquivos

import pandas as pd
from pathlib import Path


def tipo_decimal(texto):
    """Classifica um valor em texto pelo separador decimal."""
    if pd.isna(texto):
        return "vazio"
    t = str(texto).replace("R$", "").replace(" ", "").strip()
    posicoes = [i for i, c in enumerate(t) if c in ".,"]
    if not posicoes:
        return "sem separador"          # ex.: 50 ou -1200
    ultimo = posicoes[-1]
    sep = t[ultimo]
    casas = len(t) - ultimo - 1         # quantos dígitos depois do último separador
    if sep == ",":
        return "vírgula (BR)"
    if casas == 3:
        return "ambíguo (ponto + 3 dígitos)"
    return "ponto (US)"


def verificar_decimais(pasta, skiprows=4, colunas=("Valor", "Saldo")):
    """Para cada CSV da pasta, conta quantos valores usam cada formato de decimal."""
    resultados = []
    for arquivo in sorted(Path(pasta).glob("*.csv")):
        df = pd.read_csv(arquivo, skiprows=skiprows, sep=";", encoding="latin-1", dtype=str)
        # compara tudo em minúsculas, dos dois lados
        alvo = [c for c in df.columns if any(p.lower() in c.lower() for p in colunas)]
        for col in alvo:
            contagem = df[col].apply(tipo_decimal).value_counts()
            for tipo, qtd in contagem.items():
                resultados.append({"arquivo": arquivo.name, "coluna": col,
                                   "formato": tipo, "qtd": qtd})

    tabela = pd.DataFrame(resultados)
    if tabela.empty:
        print("Nenhuma coluna de valor/saldo encontrada. Confira a pasta e os nomes das colunas.")
        return tabela
    return tabela.pivot_table(index=["arquivo", "coluna"], columns="formato",
                              values="qtd", fill_value=0)

In [77]:
pasta = Path(path)
resumo = verificar_decimais(pasta)
print(resumo.to_string())

#todos os formatos com arquivo brasileiro


formato                               vírgula (BR)
arquivo                       coluna              
Extrato PagBank fevereiro.csv Saldo          114.0
                              Valor          114.0
extrato_abr25.csv             Saldo          159.0
                              Valor          159.0
extrato_abr26.csv             Saldo          162.0
                              Valor          162.0
extrato_ago25.csv             Saldo          162.0
                              Valor          162.0
extrato_ago26 (1).csv         Saldo          156.0
                              Valor          156.0
extrato_ago26.csv             Saldo          156.0
                              Valor          156.0
extrato_dez25.csv             Saldo          156.0
                              Valor          156.0
extrato_fev26.csv             Saldo          145.0
                              Valor          145.0
extrato_jan25.csv             Saldo          103.0
                              V

In [78]:
#limpeza de arquivos duplicados e nulos 
import hashlib
import pandas as pd
from pathlib import Path


def verificar_extratos(pasta, skiprows=4, ignorar=("extrato_mar26_site.csv",)):
    """
    Lê todos os CSVs da pasta, remove linhas vazias e duplicadas de cada arquivo
    e deixa de fora arquivos com conteúdo idêntico a outro.
    Não altera nem apaga nada no disco.
    Retorna: (dicionário {nome_arquivo: DataFrame limpo}, DataFrame com o relatório)
    """
    dados = {}
    relatorio = []
    assinaturas = {}  # assinatura do conteúdo -> nome do primeiro arquivo com esse conteúdo

    for arquivo in sorted(Path(pasta).glob("*.csv")):
        if arquivo.name in ignorar:
            relatorio.append({"arquivo": arquivo.name, "status": "ignorado (tratado à parte)"})
            continue

        # dtype=str: nesta etapa só comparamos; a conversão vem depois do concat
        df = pd.read_csv(arquivo, skiprows=skiprows, sep=";", encoding="latin-1", dtype=str)
        linhas_lidas = len(df)

        # 1) linhas totalmente vazias -> remove
        vazias = int(df.isnull().all(axis=1).sum())
        df = df.dropna(how="all")

        # 2) linhas com algum campo vazio -> só reporta
        com_nulo = int(df.isnull().any(axis=1).sum())

        # 3) linhas duplicadas dentro do arquivo -> remove
        duplicadas = int(df.duplicated().sum())
        df = df.drop_duplicates().reset_index(drop=True)

        # 4) arquivo duplicado: compara o conteúdo ordenado
        conteudo = df.sort_values(list(df.columns)).to_csv(index=False)
        assinatura = hashlib.md5(conteudo.encode("utf-8")).hexdigest()

        if assinatura in assinaturas:
            status = f"DUPLICADO de {assinaturas[assinatura]} (fora da base)"
        else:
            assinaturas[assinatura] = arquivo.name
            dados[arquivo.name] = df
            status = "ok"

        relatorio.append({
            "arquivo": arquivo.name,
            "linhas_lidas": linhas_lidas,
            "vazias_removidas": vazias,
            "linhas_com_nulo": com_nulo,
            "duplicadas_removidas": duplicadas,
            "linhas_finais": len(df),
            "status": status,
        })

    return dados, pd.DataFrame(relatorio)

pasta = Path(path)
dados, relatorio = verificar_extratos(pasta)
print(relatorio.to_string())

                          arquivo  linhas_lidas  vazias_removidas  linhas_com_nulo  duplicadas_removidas  linhas_finais                                             status
0   Extrato PagBank fevereiro.csv         114.0               0.0              0.0                   0.0          114.0                                                 ok
1               extrato_abr25.csv         159.0               0.0              0.0                   0.0          159.0                                                 ok
2               extrato_abr26.csv         162.0               0.0              0.0                   0.0          162.0                                                 ok
3               extrato_ago25.csv         162.0               0.0              0.0                   0.0          162.0                                                 ok
4           extrato_ago26 (1).csv         156.0               0.0              0.0                   0.0          156.0                          

In [79]:
#verificaando se todas as colunas de todos os arquivos estão iguais aos de março para unificarmos
#usando março como referencia pois essa que foi tratada separadamente e está no padrão correto
colunas_mar = mar26.columns.tolist()
for nome, df in dados.items():
    if df.columns.tolist() != colunas_mar:
        print(nome, df.columns.tolist())

In [80]:
#junção dos arquivos
dados["extrato_mar26_site.csv"] = mar26
base = pd.concat(dados, names=["arquivo_origem"]).reset_index(level=0)
base

,arquivo_origem,Data,Descrição,Valor,Saldo
0,Extrato PagBank fevereiro.csv,01/02/2025,VENDAS CARTAO CREDITO - REPASSE,"260,63","11401,26"
1,Extrato PagBank fevereiro.csv,01/02/2025,VENDAS CARTAO DEBITO - REPASSE,"159,74","11561,00"
2,Extrato PagBank fevereiro.csv,01/02/2025,VENDAS PIX MAQUININHA - REPASSE,"436,33","11997,33"
3,Extrato PagBank fevereiro.csv,01/02/2025,PIX RECEBIDO - ALINE VIEIRA,"265,00","12262,33"
4,Extrato PagBank fevereiro.csv,02/02/2025,VENDAS CARTAO CREDITO - REPASSE,"245,60","12507,93"
...,...,...,...,...,...
147,extrato_mar26_site.csv,2026-03-31,VENDAS PIX MAQUININHA - REPASSE,"423,30",0
148,extrato_mar26_site.csv,2026-03-31,PIX RECEBIDO - DIEGO MACHADO,"455,00",0
149,extrato_mar26_site.csv,2026-03-31,PIX RECEBIDO - GABRIELA NUNES,"235,00",0
150,extrato_mar26_site.csv,2026-03-31,PIX RECEBIDO - BRUNO TAVARES,"125,00",0


In [81]:
#TRANSFORMAÇÃO DOS TIPOS DAS BASE
#Toda a base esta como string, agora será transformada o tipo de cada coluna
def converter_tipos(df):
    df = df.copy()

    # Data: texto dd/mm/aaaa -> datetime
    df["Data"] = pd.to_datetime(df["Data"].astype(str).str.strip(),
                                format="%d/%m/%Y", errors="coerce")

    # Valor e Saldo: texto no formato BR -> número
    for col in ["Valor", "Saldo"]:
        texto = (df[col].astype(str).str.strip()
                         .str.replace("R$", "", regex=False)
                         .str.replace(" ", "", regex=False)
                         .str.replace(".", "", regex=False)    # separador de milhar
                         .str.replace(",", ".", regex=False))  # vírgula decimal
        df[col] = pd.to_numeric(texto, errors="coerce")

    # Falhas: linhas em que havia texto, mas a conversão não funcionou
    print("Falhas de conversão por coluna:")
    print(df[["Data", "Valor", "Saldo"]].isna().sum())
    return df


base_bruta = base.copy()          # cópia em texto, para comparar se algo falhar
base = converter_tipos(base)


Falhas de conversão por coluna:
Data     152
Valor      0
Saldo      0
dtype: int64


In [82]:
#verificação de qul arquivo deu erro    
falhou = base["Data"].isna()
print(base.loc[falhou, "arquivo_origem"].value_counts())

# o "desenho" das datas que falharam (dígitos viram 9)
print(base_bruta.loc[falhou, "Data"].str.replace(r"\d", "9", regex=True).value_counts())
print(base_bruta.loc[falhou, "Data"].head())

arquivo_origem
extrato_mar26_site.csv    152
Name: count, dtype: int64
Data
9999-99-99    152
Name: count, dtype: int64
0    2026-03-01
1    2026-03-01
2    2026-03-01
3    2026-03-01
4    2026-03-02
Name: Data, dtype: str


In [83]:
#arrumando o tipo  para conversão de data do arquivo de março que estava com o formato diferente dos outros
base.loc[falhou, "Data"] = pd.to_datetime(base_bruta.loc[falhou, "Data"].str.strip(),
                                          format="%Y-%m-%d")   # ajuste aqui

print(base["Data"].isna().sum())   # tem que dar 0

0


In [ ]:
#verificação e a coluna continua sendo data
print(base["Data"].dtype)            # esperado: datetime64[ns]

#verificando se as datas de março são mesmo de março
mar = base[base["arquivo_origem"] == "extrato_mar26_site.csv"]
print(mar["Data"].min(), mar["Data"].max())

datetime64[us]
2026-03-01 00:00:00 2026-03-31 00:00:00


In [ ]:
#Verifiquei anteriormente que tem dados de 07/25 na bse de 06/25. Agora é a verificação se há masi sobreposições de datas entre os arquivos.
import numpy as np


def verificar_sobreposicao(base, chave=("Data", "Descricao", "Valor","Saldo")):
    """
    Encontra lançamentos que estão no arquivo de um mês diferente do deles
    e verifica se eles também existem no arquivo do mês correto.
    Precisa rodar DEPOIS do converter_tipos (Data como datetime).
    """
    chave = list(chave)
    base = base.copy()
    base["id_linha"] = range(len(base))                 # para excluir depois com segurança
    base["mes_lanc"] = base["Data"].dt.to_period("M")   # mês do lançamento

    # mês "dono" de cada arquivo = mês mais frequente nas datas dele
    mes_arquivo = base.groupby("arquivo_origem")["mes_lanc"].agg(lambda s: s.mode().iloc[0])
    base["mes_arquivo"] = base["arquivo_origem"].map(mes_arquivo)

    # numera lançamentos idênticos dentro de cada arquivo (1º, 2º...)
    # assim dois Pix iguais de R$ 50 no mesmo dia são comparados um a um
    base["ocorrencia"] = base.groupby(["arquivo_origem"] + chave).cumcount()

    no_lugar_certo = base[base["mes_lanc"] == base["mes_arquivo"]]
    fora_do_mes = base[base["mes_lanc"] != base["mes_arquivo"]]

    referencia = (no_lugar_certo[chave + ["ocorrencia", "arquivo_origem"]]
                  .rename(columns={"arquivo_origem": "arquivo_correto"})
                  .drop_duplicates(subset=chave + ["ocorrencia"]))

    comparacao = fora_do_mes.merge(referencia, on=chave + ["ocorrencia"],
                                   how="left", indicator=True)

    meses_com_arquivo = set(mes_arquivo.values)
    comparacao["veredito"] = np.select(
        [comparacao["_merge"] == "both",
         ~comparacao["mes_lanc"].isin(meses_com_arquivo)],
        ["encontrada no arquivo certo - pode excluir",
         "mês sem arquivo próprio - manter"],
        default="NÃO encontrada - revisar",
    )

    colunas = ["id_linha", "arquivo_origem", "mes_arquivo", "mes_lanc"] + chave + ["arquivo_correto", "veredito"]
    return comparacao[colunas], base.drop(columns=["ocorrencia"])

In [ ]:
#As linhas de juçho estão realmengte sobrepostas em junho. Pode-se excluir essa linhas
# Nenhum outro mês foi achado com sobreposição
comparacao, base = verificar_sobreposicao(base, chave=("Data", "Descrição", "Valor", "Saldo"))
print(comparacao.groupby(["arquivo_origem", "mes_lanc", "veredito"]).size().to_string())

arquivo_origem     mes_lanc  veredito                                  
extrato_jun25.csv  2025-07   encontrada no arquivo certo - pode excluir    25


In [ ]:
print(len(base))   # antes: 3161

ids_excluir = comparacao.loc[comparacao["veredito"].str.startswith("encontrada"), "id_linha"]
base = base[~base["id_linha"].isin(ids_excluir)]

print(len(base))   # tem que ter 25 linhas a menos, o valor da sobreposição de julho

# Verificart se o arquivo de junho só tem junho 
jun = base[base["arquivo_origem"] == "extrato_jun25.csv"]
print(jun["Data"].min(), jun["Data"].max())

# exclusão das linhas sobrepostas realizadas e agora o mês de junho só tem junho

3161
3136
2025-06-01 00:00:00 2025-06-30 00:00:00


In [88]:
#criação do saldo do março que esta zerado

#resetando os indices: 
base = base.reset_index(drop=True)

In [ ]:
#pegando o saldo final de fevereiro para colocar no saldo inicial de março
fev26 = base[base["arquivo_origem"] == "extrato_fev26.csv"].sort_values("id_linha")
saldo_final_fev = fev26["Saldo"].iloc[-1]
print(fev26["Data"].iloc[-1], saldo_final_fev)   #conferindo a data 

2026-02-28 00:00:00 24652.51


In [90]:
marco = base["arquivo_origem"] == "extrato_mar26_site.csv"
mar = base.loc[marco].sort_values(["Data", "id_linha"]) #coloca os lançamentos em ordem cronológica 

base.loc[mar.index, "Saldo"] = saldo_final_fev + mar["Valor"].cumsum() #soma os valores linha a linha 

In [ ]:
#comparação com o saldo de abril. O saldo final de março, temque ser a primeira linha abril - o valor lançado nela
abr = base[base["arquivo_origem"] == "extrato_abr26.csv"].sort_values("id_linha").iloc[0]
saldo_antes_abril = abr["Saldo"] - abr["Valor"]

mar = base.loc[marco].sort_values(["Data", "id_linha"])
saldo_final_mar = mar["Saldo"].iloc[-1]

print("Saldo final de março:  ", round(saldo_final_mar, 2))
print("Saldo antes de abril:  ", round(saldo_antes_abril, 2))
print("Diferença:             ", round(saldo_final_mar - saldo_antes_abril, 2))

#diferença zerada e conciliação feita

Saldo final de março:   27556.25
Saldo antes de abril:   27556.25
Diferença:              0.0


verificação de quais meses não estão na base

In [ ]:
meses_na_base = base["Data"].dt.to_period("M").unique()

todos_os_meses = pd.period_range(start=base["Data"].min(), end=base["Data"].max(), freq="M")

faltando = todos_os_meses.difference(meses_na_base)
print("Meses sem nenhum lançamento:", list(faltando))
#apenas 09/25 sem periodo 

Meses sem nenhum lançamento: [Period('2025-09', 'M')]


In [ ]:
#comparação de qtd de lançamentos por mês. Entendendo se a algum outlier
base.groupby(base["Data"].dt.to_period("M")).size()
#nao achei nada muito descrepante, mas a qtd indica que um movimento maior nos meio dos anos. Analisar com mais atenção mais para frente!

Data
2025-01    103
2025-02    114
2025-03    140
2025-04    159
2025-05    169
2025-06    154
2025-07    147
2025-08    162
2025-10    156
2025-11    155
2025-12    156
2026-01    141
2026-02    145
2026-03    152
2026-04    162
2026-05    165
2026-06    155
2026-07    154
2026-08    156
2026-09    144
2026-10    147
Freq: M, dtype: int64

Alterar tipos para fnalizar o tratamento da base

In [94]:
from pathlib import Path

pasta_tratados = Path(r"C:\Users\nadia\OneDrive\Desktop\Leticia\Documentos\Projetos\cliente_confeitaria\Analise_financeira_confeitaria\Dados\tratados")
pasta_tratados.mkdir(parents=True, exist_ok=True)

base_final = (base.drop(columns=["mes_lanc", "mes_arquivo"], errors="ignore")
                  .sort_values(["Data", "id_linha"]))

base_final.to_csv(
    pasta_tratados / "extratos_tratados.csv",
    index=False,
    sep=";",
    decimal=",",
    date_format="%d/%m/%Y",
    encoding="utf-8-sig",
)

print("Salvo em:", pasta_tratados / "extratos_tratados.csv", "| linhas:", len(base_final))

Salvo em: C:\Users\nadia\OneDrive\Desktop\Leticia\Documentos\Projetos\cliente_confeitaria\Analise_financeira_confeitaria\Dados\tratados\extratos_tratados.csv | linhas: 3136
